In [1]:
# For the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - 
# just adapt the client and the usage fields accordingly.

# Preparation
# First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure 
# everyone works with the exact same data.

# We will use gitsource for that:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. 
# Because we specify allowed_extensions={"md"}, it only checks the markdown files.


In [2]:
# We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. 
# The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

# Each file has a parse() method that returns a dictionary with its filename and content:

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1. How many lesson pages
# How many lesson pages are in the dataset?

# 24
# 72 == 72
# 240
# 720

len(documents) # 72

72

In [4]:
documents[0]["filename"] # how to extract filename

'01-agentic-rag/lessons/01-intro.md'

In [5]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

# << How does the agentic loop keep calling the model until it stops? >>

# What's the filename of the first result?

# 01-agentic-rag/lessons/03-rag.md
# 01-agentic-rag/lessons/14-agentic-loop.md == '01-agentic-rag/lessons/14-agentic-loop.md'
# 04-evaluation/lessons/13-llm-as-judge.md
# 06-best-practices/lessons/02-hybrid-search.md

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
index

In [6]:
# Now search for our question from above:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results[0]["filename"]
# '01-agentic-rag/lessons/14-agentic-loop.md'

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
# Q3. RAG
# Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

# Two solutions are possible:

# Implement the RAG flow yourself
# Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
# Build a RAG over the index from Q2 and answer the query:

# How does the agentic loop keep calling the model until it stops?

# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

# 700
# 7000 -- input_tokens=7193
# 70000
# 700000

# We count input tokens instead of price because the cost depends on the model and provider you use, 
# but the size of the prompt we send is the same for everyone.

# Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). 
# We'll read the input tokens from there.

# You will need to modify the code for the rag helper to expose the usage.

# In the RAG Helper class, llm returns only the text. Modify it to return the whole response, 
# and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

# DEPLOYING MANUALLY

def search(question):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

search_results = search(question)
# search_results  #[0]["filename"] # search works

In [9]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Q: " + question)
        lines.append("F: " + doc["filename"])
        lines.append("C: " + doc["content"])
        lines.append("")
        lines.append("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
        lines.append("")

    return "\n".join(lines).strip()

# Now we combine the question with the context into the user prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
    
# Let's try it:

prompt = build_prompt(question, search_results)

# print(prompt)
# We send it to the LLM:

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.output_text)

The agentic loop keeps calling the model with a `while True` loop and stops only when the model returns no function calls.

In the lesson, the logic is:

1. Send the current `messages` to the model.
2. Inspect `response.output`.
3. If there’s a `function_call`, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.
4. If there are no function calls this turn, `break` out of the loop.

The key exit check is:

```python
if has_function_calls == False:
    break
```

So the model is effectively in charge of deciding whether another round is needed. Your code just keeps replaying the conversation and tool outputs until the model finally responds with a normal message instead of asking for more tools.

In short:  
- model asks for tool → loop continues  
- model gives final answer with no tool calls → loop stops




In [10]:
# The usage counts tell you how many tokens the request consumed:

response.usage # input_tokens=7193

ResponseUsage(input_tokens=7193, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=196, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7389)

In [11]:
# Q4. Chunking
# The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

# gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)
# With size=2000 and step=1000 (you can see the implementation here):

# Each chunk is a window of size characters of the page.
# The window moves forward by step characters between chunks. Since step is smaller than size, consecutive chunks overlap by size - step (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
# Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).
# How many chunks do you get?

# 70
# 295 == 295
# 1100
# 4500



295

In [12]:
# Q5. RAG with chunking
# Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

# Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

# Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

# about the same
# 3× fewer -- input_tokens=2376 now vs 7193 before
# 10× fewer
# 30× fewer

chunks[10]

{'start': 4000,
 'content': "hat the LLM generates.\nThat search step is what gives the LLM the context it needs to answer\ncorrectly.\n\nWhat we just did was naive. I knew in advance which FAQ entry held the\nanswer and pasted it in by hand. What we want instead is to perform\nsearch automatically. We take the student's question, find the most\nrelevant documents, and send those to the LLM.\n\nIn code, it looks like this:\n\n```python\ndef rag(question):\n    search_results = search(question)\n    user_prompt = build_prompt(question, search_results)\n    return llm(user_prompt)\n```\n\nThat's the entire architecture. It comes down to three components.\n\nThe pieces are search, the prompt, and the LLM:\n\n- search\n- prompt\n- LLM\n\n\n```mermaid\nflowchart TD\n    U([User])\n\n    APP[Application]\n\n    DB[(DB)]\n    DOCS[[D1 ... D5]]\n\n    PROMPT[Build Prompt<br/>Question + Context]\n\n    LLM[LLM]\n\n    ANSWER([Answer])\n\n    U -->|Question| APP\n\n    APP -->|Query| DB\n    DB 

In [13]:
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)
index_chunks

In [14]:
def search_chunks(question):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index_chunks.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

search_results = search_chunks(question)
search_results[0]["filename"] # works - '01-agentic-rag/lessons/14-agentic-loop.md'

'01-agentic-rag/lessons/14-agentic-loop.md'

In [15]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Q: " + question)
        lines.append("F: " + doc["filename"])
        lines.append("C: " + doc["content"])
        lines.append("")
        lines.append("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
        lines.append("")

    return "\n".join(lines).strip()

# Now we combine the question with the context into the user prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
    
# Let's try it:

prompt = build_prompt(question, search_results)

# print(prompt)
# We send it to the LLM:

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.output_text)

The loop keeps calling the model with a `while True` loop, and it only stops when the model returns no function calls in that turn.

In the code:

- call the model
- inspect `response.output`
- if any item is a `function_call`, run the tool and set `has_function_calls = True`
- if there are no function calls, break out of the loop

So the stopping condition is simply:

```python
if has_function_calls == False:
    break
```

That means the model decides how many tool rounds are needed, and your loop just keeps going until the model gives a final answer without asking for more tools.


In [16]:
response.usage # input_tokens=2376

ResponseUsage(input_tokens=2376, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=137, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2513)

In [ ]:
# Q6. Turning it into an agent
# So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

# If you go with toyaikit:

# uv add toyaikit
# Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

# Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

# You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

# Ask it:

# How does the agentic loop work, and how is it different from plain RAG?

# The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

# How many times did the agent call search?

# Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

# 0
# 4
# 10
# 20

